# protein_complex_maps: Colab quickstart

<!-- claude: notebook written by claude; executed locally (setup cells replaced with local paths) but not yet run on Colab itself -->

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KDrewLab/protein_complex_maps/blob/py3-migration/notebooks/colab_quickstart.ipynb)

This notebook installs `protein_complex_maps` (Python 3 branch) and its external clustering tools on Google Colab, then runs a small example:

1. cluster a toy protein–protein interaction network with ClusterONE followed by MCL (`network_clustering.py`)
2. compare the predicted clusters to a gold standard of known complexes (`complex_comparison.py`)

Out of scope on this branch: the CFinder, CliXO and agglomod clustering options, the R-based `diffusion_clustering.py`, and the Flask websites.

## 1. Install the package and the clustering tools

- `protein_complex_maps` from GitHub (pulls in numpy, pandas, scipy, scikit-learn, statsmodels, biopython, ...)
- [MCL](https://micans.org/mcl/) from Ubuntu's package manager (installs `/usr/bin/mcl`)
- the [ClusterONE](https://paccanarolab.org/cluster-one/) jar (Java is already installed on Colab); its checksum is verified below

In [ ]:
!pip install -q git+https://github.com/KDrewLab/protein_complex_maps@py3-migration
!apt-get -qq update > /dev/null && apt-get -qq install -y mcl > /dev/null
!wget -q -O /content/cluster_one-1.0.jar https://paccanarolab.org/static_content/clusterone/cluster_one-1.0.jar

The clustering scripts read the tool locations from the `CLUSTERONE_JAR` and `MCL_BIN` environment variables (you can also pass `--clusterone_jar` / `--mcl_bin`).

In [ ]:
import os

os.environ["CLUSTERONE_JAR"] = "/content/cluster_one-1.0.jar"
os.environ["MCL_BIN"] = "/usr/bin/mcl"

In [ ]:
import hashlib
import subprocess

# check the downloaded ClusterONE jar is the expected file
with open(os.environ["CLUSTERONE_JAR"], "rb") as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()
assert sha256 == "3344c954df388f5cfe9795e0db375ae7189334a1eacb5bd3121507b33765a16b", "unexpected ClusterONE jar checksum: " + sha256

# check java and mcl run
print(subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
print(subprocess.run([os.environ["MCL_BIN"], "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 2. A toy interaction network

`network_clustering.py` expects one interaction per line, `id1 id2 score`, **sorted by score from highest to lowest** (it reads the list until scores drop below the threshold).

The toy network has three groups of strongly connected proteins (A–D, E–G, H–J) plus a few weak links between them.

In [ ]:
interactions = [
    ("A", "B", 0.95), ("A", "C", 0.93), ("B", "C", 0.92), ("A", "D", 0.90), ("B", "D", 0.88), ("C", "D", 0.87),
    ("E", "F", 0.86), ("E", "G", 0.84), ("F", "G", 0.83),
    ("H", "I", 0.80), ("H", "J", 0.78), ("I", "J", 0.76),
    ("D", "E", 0.30), ("G", "H", 0.25), ("C", "J", 0.20),
]
interactions.sort(key=lambda x: x[2], reverse=True)

with open("network.txt", "w") as f:
    for id1, id2, score in interactions:
        f.write("%s\t%s\t%s\n" % (id1, id2, score))

print(open("network.txt").read())

## 3. Cluster the network (ClusterONE, then MCL)

`--ppi_threshold_score 0.5` keeps interactions scoring at least 0.5. The script can sweep several thresholds and ClusterONE/MCL parameters in one run, writing one cluster file per parameter set (`clusters.ii0.txt`, `clusters.ii1.txt`, ...) and a `clusters.params` table describing each; here there is a single parameter set.

In [ ]:
import sys

!{sys.executable} -m protein_complex_maps.clustering.network_clustering --input_network network.txt --output_file clusters.txt --ppi_threshold_score 0.5 --twostep_combination clusterone mcl > clustering.log 2>&1
!cat clusters.params
print()
!cat clusters.ii0.txt

## 4. Compare the clusters to known complexes

A gold standard is a file with one complex per line, protein ids separated by spaces. Here the third known complex also contains a protein, K, that is missing from the network, so it can only be partly recovered. `ComplexComparison` computes, among others, the maximum matching ratio (MMR), its prediction-wise version (PWMMR), and Brohée & van Helden's sensitivity, PPV and accuracy.

In [ ]:
import contextlib
import io

import protein_complex_maps.evaluation.complex_comparison as cc

with open("gold_standard.txt", "w") as f:
    f.write("A B C D\nE F G\nH I J K\n")

def read_complexes(filename):
    with open(filename) as f:
        return [line.split() for line in f if line.strip()]

gold_standard = read_complexes("gold_standard.txt")
clusters = read_complexes("clusters.ii0.txt")

comparison = cc.ComplexComparison(gold_standard, clusters)
with contextlib.redirect_stdout(io.StringIO()):  # some metrics print intermediate values
    metrics = {
        "MMR": comparison.mmr(),
        "PWMMR": comparison.pwmmr(),
        "MMR/PWMMR harmonic mean": comparison.mmr_pwmmr_hmean(),
        "sensitivity": comparison.sensitivity(),
        "PPV": comparison.ppv(),
        "accuracy": comparison.acc(),
    }
for name, value in metrics.items():
    print("%-25s %.3f" % (name, value))

## 5. (Optional) Run the package's unit tests

`testClustering2` is expected to error: it uses CFinder, which is not installed. `testLength` queries the UniProt REST API, so it needs internet access.

In [ ]:
import protein_complex_maps

pkg_dir = os.path.dirname(protein_complex_maps.__file__)
!{sys.executable} -m unittest discover -b -s {pkg_dir}/test -t {os.path.dirname(pkg_dir)} 2>&1 | grep -E "^(ERROR|FAIL):|^Ran |^OK|^FAILED"